In [1]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown, update_display

from web_fetch import Website

In [19]:
Website("https://ghaza24.com").title

'غذا24- رستوران و فست فود شبانه روزی تهران'

In [58]:
load_dotenv(override=True)
GAPGPT_API_KEY = os.getenv("GAPGPT_API_KEY")
BASE_URL = "https://api.gapgpt.app/v1"
MODEL = "gpt-5.2"
openai = OpenAI(api_key="sk-TUEGtWlMBMhS7GumHG3TQWPOD7xcH1kw6MH3YLRwjs2bSUrg", base_url=BASE_URL)

In [4]:
openai

In [5]:
link_system_prompt = "You are provided with a list of links found on a Persian restaurant webpage. \
You are able to decide which of the links would be most relevant to include in the restaurant menu, \
such as links to an menu pdf file, Menu page, List, or Shop.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "menu page", "url": "https://snappfood.ir/restaurant/menu"},
        {"type": "menu page", "url": "https://ghaza24.com/shop"}
    ]
}
"""

In [18]:
web_content = Website("https://ghaza24.com/")
print(web_content.get_content())

Website title: غذا24- رستوران و فست فود شبانه روزی تهران
Website contentSkip to content
به دنبال چه هستید...
بررسی صفحه جستجو.
0
سبد
سبد خرید
حساب
تمام غذاها
صفحات
حساب من
سبد
خرید
فروشگاه
سوالات متداول
سیاست حفظ اسرار
قوانین و مقررات
تماس با ما
۴۰۴ Page
درباره ما
پیتزا
پیتزا تک نفره
پیتزا خانواده
پیش غذاها٫ سالاد و سیب زمینی
برگرها و ساندویچ ها
پاستا
سوخاری ها
کباب ها و غذاهای ایرانی
باگت ها و ساندویچ های بمبی
نوشیدنی ها
بلاگ
سفارش غذا از رستوران شبانه روزی الوشب
سفارش غذای شبانه
فست فود نزدیک من اکنون باز است
سفارش غذا شبانه شهرک غرب
سفارش غذا شهرک غرب
فست فود شبانه روزی در صادقیه تهران
سفارش فست فود شبانه روزی در گیشا
سفارش فست فود در مرزداران شبانه روزی
فست فود نزدیک به من
سفارش غذای ارزان تخفیف خورده
فست‌فود شبانه‌روزی تهران
فست‌فود شبانه‌روزی نیاوران
فست فود شبانه روزی پونک
سفارش فستفود شبانه روزی در تهران
سفارش غذا با تخفیف شبانه تهران ارزان
سفارش غذا از رستوران شبانه روزی الوشب
رستوران شبانه روزی کرج
فست فود شبانه روزی کرج
به دنبال چه هستید...
بررسی صفحه جستجو.
0
سبد
سبد خرید
ح

In [7]:
web_content.links

['#primary',
 'https://ghaza24.com/',
 'https://ghaza24.com/my-account/',
 'https://ghaza24.com/',
 'https://ghaza24.com/search/',
 '#',
 'https://ghaza24.com/my-account-2/',
 'https://ghaza24.com/cart-2/',
 'https://ghaza24.com/checkout-2/',
 'https://ghaza24.com/shop-2/',
 'https://ghaza24.com/faq/',
 'https://ghaza24.com/%d8%b3%db%8c%d8%a7%d8%b3%d8%aa-%d8%ad%d9%81%d8%b8-%d8%a7%d8%b3%d8%b1%d8%a7%d8%b1/',
 'https://ghaza24.com/%d9%82%d9%88%d8%a7%d9%86%db%8c%d9%86-%d9%88-%d9%85%d9%82%d8%b1%d8%b1%d8%a7%d8%aa/',
 'https://ghaza24.com/contact/',
 'https://borobazar.redq.io/404',
 'https://ghaza24.com/about-us/',
 '#',
 'https://ghaza24.com/product-category/%d9%be%db%8c%d8%aa%d8%b2%d8%a7/pizza/',
 'https://ghaza24.com/product-category/%d9%be%db%8c%d8%aa%d8%b2%d8%a7/%d9%be%db%8c%d8%aa%d8%b2%d8%a7-%d8%ae%d8%a7%d9%86%d9%88%d8%a7%d8%af%d9%87/',
 'https://ghaza24.com/product-category/%d9%be%db%8c%d8%b4-%d8%ba%d8%b0%d8%a7%d9%87%d8%a7%d9%ab-%d8%b3%d8%a7%d9%84%d8%a7%d8%af-%d9%88-%d8%b3%db%8c%d8%a8

In [8]:
def get_links_user_prompt(website):
    user_prompt = f"""
    Here is the link to a Persian restaurat called {website.title}: {website.url}.
    Down below there's a list of links which are extracted from this restaurant.
    Your job is to select links which are most relevant to the service's food menu section.
    Remember the response you generate should be full https URL in JSON format with no other explanatio.
    Some links might be relaive:
    {website.links}
"""
    return user_prompt

In [9]:
# def prompt_messages(sys, usr):
#    return [
#        {"role": "system", "content": sys},
#        {"role": "user", "content": usr}
#    ]

In [23]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": link_system_prompt},
        {"role": "user", "content": get_links_user_prompt(website)},
    ], response_format={"type": "json_object"})
    links = response.choices[0].message.content
    return json.loads(links)

In [27]:
def get_web_details(url):
    result = "Landing Page:\n"
    result += Website(url).get_content()
    links = get_links(url)
    print(f"Founded Links: {links}")
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_content()
    return result


In [37]:
system_prompt = "You are an assistant that analyzes the contents of several menu pages from a persian restaurant website \
and creates eye-catching and modern restaurant menu with dishes and prices in Rial. If there are no links found, Just apologize and say so. \
Note: Your respond should be right-to-left (RTL) and also in Markdown - Do not wrap Markdown in code block. \
Remember say nothing more than just the menu."

def get_restaurant_menu_user_prompt(url):
    website = Website(url)
    user_prompt = f"You are looking at a restaurant called: {website.title}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a restaurant menu in markdown.\n"
    user_prompt += get_web_details(website.url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [33]:
def create_restaurant_menu(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_restaurant_menu_user_prompt(url)},
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [62]:
create_restaurant_menu("https://ghaza24.com/")

KeyboardInterrupt: 

In [35]:
create_restaurant_menu("https://nayeb.com/")

Founded Links: {'links': [{'type': 'menu page', 'url': 'https://nayeb.com/online'}]}


<div dir="rtl">

# منوی رستوران نایب  
**سفارش آنلاین:** ۱۱:۳۰ تا ۲۱:۴۵  
**روش دریافت:** ارسال با پیک / دریافت حضوری  
**هزینه ارسال (نمایش‌داده‌شده):** ۱۰,۰۰۰ ریال  

---

## چلوکباب‌های اصیل نایب (امضای برند)
> ترکیب برنج ایرانی، کره و کباب؛ به سبک قدیمی نایب

| آیتم | توضیح کوتاه | قیمت (ریال) |
|---|---|---:|
| چلوکباب کوبیده نایب | دو سیخ کوبیده + برنج ایرانی + کره | ۲,۹۸۰,۰۰۰ |
| چلوکباب برگ | گوشت راسته گریل‌شده + برنج ایرانی + کره | ۴,۹۸۰,۰۰۰ |
| چلوکباب سلطانی | یک سیخ برگ + یک سیخ کوبیده + برنج + کره | ۵,۶۸۰,۰۰۰ |
| چلوجوجه کباب زعفرانی | جوجه زعفرانی گریل + برنج + کره | ۳,۶۸۰,۰۰۰ |

---

## پلوها و غذاهای ایرانی
| آیتم | توضیح کوتاه | قیمت (ریال) |
|---|---|---:|
| باقالی‌پلو با گوشت | باقالی‌پلو زعفرانی + گوشت | ۴,۸۸۰,۰۰۰ |
| زرشک‌پلو با مرغ | مرغ زعفرانی + زرشک‌پلو | ۳,۳۸۰,۰۰۰ |

---

## پیش‌غذا و مخلفات
| آیتم | توضیح کوتاه | قیمت (ریال) |
|---|---|---:|
| ماست و خیار | سنتی و خنک | ۷۸۰,۰۰۰ |
| سالاد شیرازی | تازه و کلاسیک | ۶۸۰,۰۰۰ |
| زیتون پرورده | شمالی و خوش‌طعم | ۹۸۰,۰۰۰ |
| ترشی مخلوط | کنار غذای ایرانی | ۵۸۰,۰۰۰ |
| نان تازه | مناسب کنار کباب | ۳۸۰,۰۰۰ |

---

## نوشیدنی
| آیتم | قیمت (ریال) |
|---|---:|
| دوغ | ۶۸۰,۰۰۰ |
| نوشابه | ۶۸۰,۰۰۰ |
| آب معدنی | ۴۸۰,۰۰۰ |

---

### راهنمای سفارش
- برای ادامه سفارش، **شعبه** و **آدرس** را انتخاب کنید.  
- توجه: عدم تطابق موقعیت مکانی با آدرس می‌تواند باعث افزایش زمان و هزینه ارسال شود.

</div>

In [38]:
create_restaurant_menu("https://snappfood.ir/")

Founded Links: {'links': []}


متأسفم، در اطلاعات ارسالی هیچ لینک یا محتوای منویی از صفحات رستوران وجود ندارد تا بتوانم بر اساس آن منو تهیه کنم.

In [49]:
create_restaurant_menu("https://pizzatoranj.com/")

Founded Links: {'links': [{'type': 'menu page', 'url': 'https://pizzatoranj.com/s'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168689-%D9%BE%DB%8C%D8%AA%D8%B2%D8%A7-%D8%A7%DB%8C%D8%AA%D8%A7%D9%84%DB%8C%D8%A7%DB%8C%DB%8C'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168686-%D9%BE%D8%A7%D8%B3%D8%AA%D8%A7'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168688-%D9%BE%DB%8C%D8%AA%D8%B2%D8%A7'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168691-%D8%BA%D8%B0%D8%A7%DB%8C-%D8%A7%D8%B5%D9%84%DB%8C'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168690-%D8%B3%D8%A7%D9%86%D8%AF%D9%88%DB%8C%DA%86-%D9%87%D8%A7'}, {'type': 'category', 'url': 'https://pizzatoranj.com/s/categories/168687-%D8%A8%D8%B4%D9%82%D8%A7%D8%A8-%D9%87%D8%A7'}]}


<div dir="rtl">

# منوی پیتزا ترنج (ترنج)

## پیتزا ایتالیایی سنگی (۳۲ سانتی)
| آیتم | ترکیبات | قیمت (ریال) |
|---|---|---:|
| پیتزا سنت لوئیس | سس گوجه، سالامی، فلفل دلمه، زیتون، پیاز | ۷,۵۰۰,۰۰۰ |
| پیتزا وجترین | سس گوجه، پیاز، فلفل دلمه، قارچ، زیتون، پنیر میکس | ۵,۷۰۰,۰۰۰ |
| پیتزا مخصوص ایتالیایی | سس گوجه، بیکن، راسته گوساله، پپرونی، زیتون، پنیر میکس | ۹,۳۵۰,۰۰۰ |
| پیتزا رست بیف | ران گوساله رست‌شده طعم‌دار، قارچ، فلفل دلمه | ۹,۰۳۰,۰۰۰ |
| بولونیزه پستو | راسته گوساله چرخ‌شده، قارچ، سالامی، سس پستو، پنیر پارمسان | ۸,۷۰۰,۰۰۰ |
| پیتزا سیر و استیک | راسته گوساله، سس سیر، پیاز، پنیر پارمسان | ۹,۴۴۰,۰۰۰ |
| استیک پستو | راسته گوساله، پنیر پارمسان، سس پستو | ۹,۳۵۰,۰۰۰ |
| پیتزا استیک آلفردو | راسته گوساله و پیاز، پنیر پارمسان، سس آلفردو | ۹,۳۵۰,۰۰۰ |
| پیتزا چیکن آلفردو | مرغ گریل‌شده، فلفل دلمه، قارچ، سس آلفردو | ۷,۷۲۰,۰۰۰ |
| پیتزا وگا ایتالیایی | ژامبون گوشت و مرغ، فلفل دلمه، قارچ، کنجد | ۷,۰۴۰,۰۰۰ |
| پیتزا تاکینو پستو | استیک بوقلمون فرآوری‌شده دودی، سس پستو، پنیر پارمسان، کنجد، موزارلا | ۸,۳۰۰,۰۰۰ |
| پیتزا مارگاریتا (ایتالیایی) | پنیر موزارلا، گوجه | ۵,۱۵۰,۰۰۰ |
| پیتزا آل پولو (ایتالیایی) | مرغ گریل‌شده، قارچ، فلفل پاپریکا | ۷,۳۵۰,۰۰۰ |
| پیتزا پپرونی (ایتالیایی) | پپرونی، زیتون سیاه | ۷,۲۵۰,۰۰۰ |
| پیتزا پرشوتو (ایتالیایی) | بیکن، سالامی، گوجه، فلفل پاپریکا | ۷,۵۷۰,۰۰۰ |
| پیتزا مامامیا (ایتالیایی) | راسته گوساله، مرغ گریل‌شده، قارچ، پیاز قرمز | ۸,۳۰۰,۰۰۰ |
| پیتزا سیسیل (ایتالیایی) | پپرونی، مرغ گریل‌شده، پیاز قرمز | ۷,۶۰۰,۰۰۰ |
| پیتزا کاپاچو (ایتالیایی) | — | ۹,۱۰۰,۰۰۰ |

## سس‌ها
| آیتم | توضیحات | قیمت (ریال) |
|---|---|---:|
| سس ایولی | سس ایولی مخصوص پیتزا ایتالیایی | ۲۵۰,۰۰۰ |
| سس ایولی پیتزا ایتالیایی | — | ۲۵۰,۰۰۰ |

</div>